In [1]:
import os, random, pickle, time
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression


def seed_all(seed: int = 20) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    # TensorFlow removed; no tf.random.set_seed needed


seed_all(20)




In [2]:
TRAIN_PATH = "/kaggle/input/osic-pulmonary-fibrosis-progression/train.csv"
TEST_META_PATH = "/kaggle/input/osic-pulmonary-fibrosis-progression/test.csv"
SUBMIT_TEMPLATE_PATH = (
    "/kaggle/input/osic-pulmonary-fibrosis-progression/sample_submission.csv"
)

train_df = pd.read_csv(TRAIN_PATH)
test_meta_df = pd.read_csv(TEST_META_PATH)
sub_template = pd.read_csv(SUBMIT_TEMPLATE_PATH)




In [3]:
combined = pd.concat(
    [train_df[["Sex", "SmokingStatus"]], test_meta_df[["Sex", "SmokingStatus"]]], axis=0
)

le_sex = LabelEncoder().fit(combined["Sex"])
le_smoke = LabelEncoder().fit(combined["SmokingStatus"])


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["Sex_enc"] = le_sex.transform(df["Sex"])
    df["Smoke_enc"] = le_smoke.transform(df["SmokingStatus"])
    return df[["Weeks", "Percent", "Age", "Sex_enc", "Smoke_enc"]]


X = add_features(train_df)
y = train_df["FVC"].values.astype(np.float32)




In [4]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)




In [5]:
def build_model(input_dim: int):
    """Linear regression model (scikit‑learn)."""
    model = LinearRegression()
    return model


model = build_model(X_train.shape[1])
model.fit(X_train, y_train)




LinearRegression()

In [6]:
EPOCHS = 30  # retained for compatibility; not used by sklearn
BATCH_SIZE = 128  # retained for compatibility; not used by sklearn

# No additional training loop needed; model already fitted




In [7]:
val_pred = model.predict(X_val)
mae_val = mean_absolute_error(y_val, val_pred)
CONFIDENCE = max(70.0, mae_val)




In [8]:
sub_template["Patient"] = sub_template["Patient_Week"].str.split("_").str[0]
sub_template["Week"] = sub_template["Patient_Week"].str.split("_").str[1].astype(int)

sub_merged = sub_template.merge(
    test_meta_df,
    how="left",
    left_on="Patient",
    right_on="Patient",
    suffixes=("", "_meta"),
)

sub_features = add_features(sub_merged)




In [9]:
sub_pred = model.predict(sub_features)

submission = pd.DataFrame(
    {
        "Patient_Week": sub_template["Patient_Week"],
        "FVC": np.clip(sub_pred, a_min=0, a_max=None),  # ensure non‑negative
        "Confidence": CONFIDENCE,
    }
)

submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)

print(f"Submission written to {submission_path}")
print(f"Validation MAE (used as confidence): {mae_val:.2f}")

Submission written to submission.csv
Validation MAE (used as confidence): 263.27
